# 03 - Dummy baselines (v2)

Goal: measure what a model that **ignores all features** scores on our metrics. Every real model must beat these numbers.

- `DummyClassifier(strategy="prior")`: predicts the >50K share of the training part as the probability for everyone. At threshold 0.5 it always predicts <=50K (the majority class).
- `DummyClassifier(strategy="constant", constant=1)`: predicts >50K for everyone (probability 1).
- CV: the 5 folds of `fold_r0` from `results/cv_folds.csv`. Training rows only; the test rows are never loaded into the analysis.
- Unweighted (no `fnlwgt`), no resampling.
- v2: built on the de-duplicated split (`data_version = v2_dedup` in the log). v1 is archived in `notebooks/archive_v1/` and `results/archive_v1/`.

## 1. Kernel and library versions
`sys.executable` should be `C:\Users\User\venvs\classification\Scripts\python.exe` (kernel "Python (classification)"). If not, stop and switch the kernel.

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)

print('Python executable:', sys.executable)
print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)

Python executable: c:\Users\User\venvs\classification\Scripts\python.exe
pandas       3.0.6
numpy        2.4.6
scikit-learn 1.9.1


## 2. Load data, split and folds (read-only)
- Same `read_csv` arguments as before: `header=None`, the 15 names, `keep_default_na=False`.
- The v2 split lists only **kept** rows: its `row_id`s must be exactly all positions 0 ... 48,841 minus the `row_id`s in `dropped_duplicates.csv` (48,790 entries). This catches a mix-up with v1 files early.
- Train and test sizes are read from the split file and printed, not compared with fixed numbers.
- `cv_folds.csv` must contain exactly the train `row_id`s, sorted, so no test row can enter.
- Train rows are taken sorted by `row_id`, which lines them up with `cv_folds.csv` row by row. The full raw table is then deleted.
- Target: `y = 1` for >50K, `0` for <=50K. `sex` is kept only for reporting metrics by sex; it is not a feature here (the dummies use no features).

In [2]:
ROOT = Path.cwd()
if not (ROOT / 'data' / 'adult.csv').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'data' / 'adult.csv'
RESULTS_PATH = ROOT / 'results'
SPLIT_PATH = RESULTS_PATH / 'split_indices.csv'
DROPPED_PATH = RESULTS_PATH / 'dropped_duplicates.csv'
FOLDS_PATH = RESULTS_PATH / 'cv_folds.csv'
for path in [DATA_PATH, SPLIT_PATH, DROPPED_PATH, FOLDS_PATH]:
    assert path.exists(), f'Not found: {path}'
print('Project root:', ROOT)

COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income',
]
df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS, keep_default_na=False)
assert df.shape == (48842, 15), f'Unexpected shape {df.shape}'

# Split file (v2: kept rows only)
split_df = pd.read_csv(SPLIT_PATH)
assert list(split_df.columns) == ['row_id', 'split'], f'Unexpected columns {list(split_df.columns)}'
assert split_df['row_id'].is_unique, 'row_id has duplicates'
split_labels = set(split_df['split'].unique())
assert split_labels == {'train', 'test'}, f'Unexpected split labels {split_labels}'

# Coverage: all positions minus the dropped duplicates
dropped_df = pd.read_csv(DROPPED_PATH)
assert list(dropped_df.columns) == ['row_id', 'kept_row_id'], f'Unexpected columns {list(dropped_df.columns)}'
assert dropped_df['row_id'].is_unique, 'dropped row_id has duplicates'
expected_ids = set(range(len(df))) - set(dropped_df['row_id'])
assert set(split_df['row_id']) == expected_ids, 'Split row_ids are not exactly all positions minus the dropped duplicates'
assert len(split_df) == 48790, f'Split has {len(split_df)} rows, expected 48790'

split_counts = split_df['split'].value_counts()
N_TRAIN = int(split_counts['train'])
N_TEST = int(split_counts['test'])
print(f'Train rows: {N_TRAIN:,}')
print(f'Test rows:  {N_TEST:,} (not used here)')

train_ids = np.sort(split_df.loc[split_df['split'] == 'train', 'row_id'].to_numpy())

# Folds file: exactly the train row_ids, sorted
FOLD_COLS = ['fold_r0', 'fold_r1', 'fold_r2']
folds = pd.read_csv(FOLDS_PATH)
assert list(folds.columns) == ['row_id'] + FOLD_COLS, f'Unexpected columns {list(folds.columns)}'
assert np.array_equal(folds['row_id'].to_numpy(), train_ids), 'cv_folds.csv row_ids differ from the train row_ids'

# Keep only train rows; same order as folds (both sorted by row_id)
train_df = df.iloc[train_ids].reset_index(drop=True)
del df
assert len(train_df) == N_TRAIN

assert set(train_df['income'].unique()) == {'<=50K', '>50K'}
assert set(train_df['sex'].unique()) == {'Male', 'Female'}

y = (train_df['income'] == '>50K').astype(int).to_numpy()
sex = train_df['sex'].to_numpy()
fold = folds['fold_r0'].to_numpy()
assert set(np.unique(fold)) == set(range(5))

print(f'>50K share (train): {y.mean():.4%}')

Project root: c:\Users\User\OneDrive\Desktop\Classification_Model
Train rows: 39,032
Test rows:  9,758 (not used here)
>50K share (train): 23.9419%


## 3. Out-of-fold predictions
For each fold k = 0..4 of `fold_r0`: fit on the rows **not** in fold k, predict the rows in fold k. After 5 folds, every train row has exactly one prediction made by a model that never saw it: the **out-of-fold (OOF)** predictions.
- The feature matrix is a single column of zeros: the dummies ignore features anyway.
- `clone` gives a fresh, unfitted model for every fold.
- Labels at threshold 0.5: `pred = 1` if `p >= 0.5`, else 0.

In [3]:
DATA_VERSION = 'v2_dedup'
THRESHOLD = 0.5
N_SPLITS = 5
X_dummy = np.zeros((len(y), 1))

BASELINES = {
    'baseline_prior_r0': {
        'estimator': DummyClassifier(strategy='prior'),
        'params': 'strategy=prior',
        'notes': 'predicts training-part >50K share for everyone',
    },
    'baseline_constant1_r0': {
        'estimator': DummyClassifier(strategy='constant', constant=1),
        'params': 'strategy=constant, constant=1',
        'notes': 'predicts >50K for everyone',
    },
}

oof_p = {}
oof_pred = {}
for exp_id, spec in BASELINES.items():
    p = np.full(len(y), np.nan)
    for k in range(N_SPLITS):
        is_tr = fold != k
        is_va = fold == k
        model = clone(spec['estimator'])
        model.fit(X_dummy[is_tr], y[is_tr])
        assert list(model.classes_) == [0, 1]
        p[is_va] = model.predict_proba(X_dummy[is_va])[:, 1]
    assert not np.isnan(p).any(), f'{exp_id}: some rows got no OOF prediction'
    oof_p[exp_id] = p
    oof_pred[exp_id] = (p >= THRESHOLD).astype(int)
    print(exp_id, '- distinct OOF probabilities:', np.unique(p.round(6)))

baseline_prior_r0 - distinct OOF probabilities: [0.239416 0.239424]
baseline_constant1_r0 - distinct OOF probabilities: [1.]


## 4. Metrics
- **Per fold:** PR-AUC (`average_precision_score`) on each validation fold, then mean and std across the 5 folds (std with `ddof=1`, the sample std).
- **Pooled OOF:** all train OOF predictions together: PR-AUC, ROC-AUC, accuracy, precision, recall, F1 (`zero_division=0`) and the confusion matrix, overall and by sex.
- Expectation: PR-AUC close to the >50K share and ROC-AUC close to 0.5 for both, because the scores do not rank anyone above anyone else. For the prior model the pooled values can differ slightly from that, because each fold's training part has a slightly different >50K share, so pooled scores are not perfectly constant. Per-fold PR-AUC avoids this.

In [4]:
def pooled_metrics(y_true, p, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        'pr_auc': average_precision_score(y_true, p),
        'roc_auc': roc_auc_score(y_true, p),
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

SEXES = ['Female', 'Male']
results = {}

for exp_id in BASELINES:
    p = oof_p[exp_id]
    pred = oof_pred[exp_id]

    per_fold = np.array([
        average_precision_score(y[fold == k], p[fold == k]) for k in range(N_SPLITS)
    ])
    overall = pooled_metrics(y, p, pred)
    by_sex = {s: pooled_metrics(y[sex == s], p[sex == s], pred[sex == s]) for s in SEXES}
    results[exp_id] = {
        'pr_auc_mean': per_fold.mean(),
        'pr_auc_std': per_fold.std(ddof=1),
        'overall': overall,
        'by_sex': by_sex,
    }

    print('=' * 60)
    print(exp_id)
    print('=' * 60)
    print('PR-AUC per fold:', np.round(per_fold, 4))
    print(f'PR-AUC mean: {per_fold.mean():.4f}   std: {per_fold.std(ddof=1):.4f}')
    print()

    table = pd.DataFrame({'overall': overall, **by_sex})
    print('Pooled OOF metrics (threshold 0.5):')
    print(table.round(4).to_string())
    print()

    for name, m in [('overall', overall)] + list(by_sex.items()):
        cm = pd.DataFrame(
            [[m['tn'], m['fp']], [m['fn'], m['tp']]],
            index=['true <=50K', 'true >50K'],
            columns=['pred <=50K', 'pred >50K'],
        )
        print(f'Confusion matrix ({name}):')
        print(cm.to_string())
        print()

baseline_prior_r0
PR-AUC per fold: [0.2394 0.2394 0.2394 0.2394 0.2394]
PR-AUC mean: 0.2394   std: 0.0000

Pooled OOF metrics (threshold 0.5):
              overall      Female        Male
pr_auc         0.2394      0.1093      0.3039
roc_auc        0.5000      0.5000      0.5000
accuracy       0.7606      0.8907      0.6961
precision      0.0000      0.0000      0.0000
recall         0.0000      0.0000      0.0000
f1             0.0000      0.0000      0.0000
tn         29687.0000  11526.0000  18161.0000
fp             0.0000      0.0000      0.0000
fn          9345.0000   1415.0000   7930.0000
tp             0.0000      0.0000      0.0000

Confusion matrix (overall):
            pred <=50K  pred >50K
true <=50K       29687          0
true >50K         9345          0

Confusion matrix (Female):
            pred <=50K  pred >50K
true <=50K       11526          0
true >50K         1415          0

Confusion matrix (Male):
            pred <=50K  pred >50K
true <=50K       18161        

## 5. Append to the experiment log
One row per baseline in `results/experiment_log.csv` (38 columns). The file is created with a header if missing; the v1 log is archived, so v2 starts with an empty log.
- `data_version` (after `experiment_id`) records which data the row was computed on: `v2_dedup` here.
- If the file exists, its header must match the columns below exactly; otherwise we stop instead of writing a mismatched row.
- A row is **skipped** (a notice is printed) if the same (`experiment_id`, `data_version`) pair is already in the log, so re-running this notebook does not create duplicates, while a future data version would still be logged.

In [5]:
LOG_PATH = RESULTS_PATH / 'experiment_log.csv'
METRICS = ['pr_auc', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1']
LOG_COLUMNS = (
    ['timestamp', 'experiment_id', 'data_version', 'notebook', 'model', 'params',
     'features', 'imbalance', 'sample_weight', 'cv_scheme', 'n_rows', 'threshold',
     'pr_auc_mean', 'pr_auc_std', 'pr_auc_pooled',
     'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'tn', 'fp', 'fn', 'tp']
    + [f'{m}_female' for m in METRICS]
    + [f'{m}_male' for m in METRICS]
    + ['sklearn_version', 'notes']
)
assert len(LOG_COLUMNS) == 38, f'Expected 38 log columns, got {len(LOG_COLUMNS)}'

timestamp = datetime.now().isoformat(timespec='seconds')
rows = []
for exp_id, spec in BASELINES.items():
    r = results[exp_id]
    o = r['overall']
    row = {
        'timestamp': timestamp,
        'experiment_id': exp_id,
        'data_version': DATA_VERSION,
        'notebook': '03_baseline.ipynb',
        'model': type(spec['estimator']).__name__,
        'params': spec['params'],
        'features': 'none (dummy zeros)',
        'imbalance': 'none',
        'sample_weight': 'none',
        'cv_scheme': 'fold_r0, 5-fold',
        'n_rows': len(y),
        'threshold': THRESHOLD,
        'pr_auc_mean': r['pr_auc_mean'],
        'pr_auc_std': r['pr_auc_std'],
        'pr_auc_pooled': o['pr_auc'],
        'sklearn_version': sklearn.__version__,
        'notes': spec['notes'],
    }
    for key in ['roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'tn', 'fp', 'fn', 'tp']:
        row[key] = o[key]
    for s in SEXES:
        for m in METRICS:
            row[f'{m}_{s.lower()}'] = r['by_sex'][s][m]
    rows.append(row)

new_rows = pd.DataFrame(rows, columns=LOG_COLUMNS)
assert not new_rows.isna().any().any(), 'Some log fields are empty'

log_exists = LOG_PATH.exists()
if log_exists:
    existing = pd.read_csv(LOG_PATH)
    assert list(existing.columns) == LOG_COLUMNS, 'Existing log header differs from LOG_COLUMNS; not writing.'
    existing_keys = set(zip(existing['experiment_id'], existing['data_version']))
else:
    existing_keys = set()

is_new = np.array([
    (e, d) not in existing_keys
    for e, d in zip(new_rows['experiment_id'], new_rows['data_version'])
])
for exp_id in new_rows.loc[~is_new, 'experiment_id']:
    print(f'Already in log for {DATA_VERSION}, skipped:', exp_id)

to_add = new_rows[is_new]
if len(to_add) > 0:
    to_add.to_csv(LOG_PATH, mode='a', header=not log_exists, index=False)

print('Log file:', LOG_PATH)
print('Rows added:', len(to_add))
print('Rows in log now:', len(pd.read_csv(LOG_PATH)) if LOG_PATH.exists() else 0)

Log file: c:\Users\User\OneDrive\Desktop\Classification_Model\results\experiment_log.csv
Rows added: 2
Rows in log now: 2
